In [32]:
import pandas as pd
import numpy as np
pd.options.display.max_rows = 1000
import matplotlib.pyplot as plt
from sklearn.metrics import make_scorer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.model_selection import permutation_test_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import pickle
import os
from collections import Counter

import sys
sys.path.append("../../code")
import interpretability

In [33]:
response_var = "responder"
response_var_cap = "Responder"
conditions = "mdd_bp"
conditions_cap = "MDD Bandpower"
eval_metric = "nppv"
eval_metric_cap = "nPPV"

In [34]:
# load in dynamix data
df_dynamix = pd.read_pickle('../../data/dynamix_ec_wexp_all.pkl')
print("Dynamix dataframe shape:", df_dynamix.shape) # 128 subjects, 1 + 80 (num_weights)
print(df_dynamix.head())

Dynamix dataframe shape: (120, 81)
  participants_ID  dynamix_ec_std_1  dynamix_ec_std_2  dynamix_ec_std_3  \
0    sub-87999321      3.462437e-07          0.006754          0.028750   
1    sub-88000181      1.648173e-07          0.004078          0.016838   
2    sub-88000313      3.955960e-07          0.004775          0.017861   
3    sub-88000489      7.441853e-06          0.005344          0.022563   
4    sub-88000533      5.929624e-07          0.006812          0.025632   

   dynamix_ec_std_4  dynamix_ec_std_5  dynamix_ec_std_6  dynamix_ec_std_7  \
0          0.007361          0.003901      4.248577e-07          0.007051   
1          0.007159          0.003970      1.753971e-07          0.008290   
2          0.012186          0.004588      4.532127e-07          0.010270   
3          0.004421          0.002084      1.091409e-05          0.004905   
4          0.020051          0.010480      5.707828e-07          0.019658   

   dynamix_ec_std_8  dynamix_ec_std_9  ...  dynamix

### Merge dataframes

In [35]:
demographic_df = pd.read_csv(f"../../data/final_dataset_{response_var}.csv")
demographic_df.head()
print("Demographic dataframe shape:", demographic_df.shape)

duplicate_ids = demographic_df["participants_ID"].value_counts()
duplicate_ids = duplicate_ids[duplicate_ids > 1].index
demographic_df = demographic_df.drop_duplicates(subset="participants_ID", keep="first")
print("Shape after keeping only first entry for each participant for demographic_df:", demographic_df.shape)

# df = pd.merge(bandpower_top_few_df, demographic_df, on='participants_ID', how='inner')
df = pd.merge(df_dynamix, demographic_df, on='participants_ID', how='inner')
print("Merged dataframe shape:", df.shape)
print(df.head())

Demographic dataframe shape: (230, 114)
Shape after keeping only first entry for each participant for demographic_df: (215, 114)
Merged dataframe shape: (120, 194)
  participants_ID  dynamix_ec_std_1  dynamix_ec_std_2  dynamix_ec_std_3  \
0    sub-87999321      3.462437e-07          0.006754          0.028750   
1    sub-88000181      1.648173e-07          0.004078          0.016838   
2    sub-88000313      3.955960e-07          0.004775          0.017861   
3    sub-88000489      7.441853e-06          0.005344          0.022563   
4    sub-88000533      5.929624e-07          0.006812          0.025632   

   dynamix_ec_std_4  dynamix_ec_std_5  dynamix_ec_std_6  dynamix_ec_std_7  \
0          0.007361          0.003901      4.248577e-07          0.007051   
1          0.007159          0.003970      1.753971e-07          0.008290   
2          0.012186          0.004588      4.532127e-07          0.010270   
3          0.004421          0.002084      1.091409e-05          0.004905   


### Preprocessing & splitting

In [36]:
cols = ['age', 'gender', 'BDI_pre']
dynamix_cols = df.filter(regex='dynamix$').columns.tolist()

# construct X
X = df[cols + dynamix_cols]
y = df[response_var_cap]

# binary vars float --> str
X = X.copy()
X['gender'] = X['gender'].map({1.0: 'male', 0.0: 'female'})

print("X shape:", X.shape)
print("X columns:", X.columns)
# print("X column datatypes:", X.dtypes)
print(X.head())

X shape: (120, 3)
X columns: Index(['age', 'gender', 'BDI_pre'], dtype='object')
     age  gender  BDI_pre
0  49.66    male     20.0
1  45.99  female     47.0
2  35.38  female     21.0
3  42.36    male     21.0
4  45.14  female     42.0


In [37]:
# construct preprocesser
# decide which encoder to use on each feature
onehot_ftrs = ['gender']
std_ftrs = ['age', 'BDI_pre'] + dynamix_cols

# one hot pipeline
# X['ever_used_drugs'] = X['ever_used_drugs'].fillna(99)
onehot_transformer = Pipeline(steps=[
    ('imputer0', SimpleImputer(strategy='constant')), 
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
])

# numeric pipeline with imputation + scaling
numeric_transformer = Pipeline(steps=[
    ('imputer2', SimpleImputer(strategy='median')),  # fills missing values with median
    ('scaler', StandardScaler())                     # scales features
])

# collect all the encoders
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_transformer, onehot_ftrs), 
        ('std', numeric_transformer, std_ftrs)])

clf = Pipeline(steps=[('preprocessor', preprocessor)])

In [38]:
# outer split: stratified split test and other
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42) 

print(X_train_val.shape)
print(X_test.shape)

# inner split: stratified k fold with k = 4 (60-20-20)
inner_split = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(inner_split.split(X_train_val, y_train_val)):
    print(f"Fold {fold+1}")
    X_train, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    print("X_train shape:", X_train.shape)
    print("X_val shape:", X_val.shape)

(96, 3)
(24, 3)
Fold 1
X_train shape: (72, 3)
X_val shape: (24, 3)
Fold 2
X_train shape: (72, 3)
X_val shape: (24, 3)
Fold 3
X_train shape: (72, 3)
X_val shape: (24, 3)
Fold 4
X_train shape: (72, 3)
X_val shape: (24, 3)


### Pipeline

In [39]:
# construct custom nppv scoring function
def nppv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    denom_ppv = tp + fp
    percent_pos = np.mean(y_true == 1)

    if denom_ppv == 0 or percent_pos == 0:
        return 0
        
    ppv = tp / denom_ppv
    return (ppv / percent_pos - 1) * 100

nppv_scorer = make_scorer(nppv)

In [ ]:
def MLpipe_shuffle_cv(X, y, preprocessor, ML_algo, num_folds=1000, test_size=0.1):
    cv_split = StratifiedShuffleSplit(n_splits=num_folds, test_size=test_size, random_state=42)

    train_scores = []
    val_scores = []
    nPPVs = []

    remission_rate = np.mean(y)

    for fold, (train_idx, val_idx) in enumerate(cv_split.split(X, y)):
        pipeline = make_pipeline(preprocessor, ML_algo)
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])

        train_score = pipeline.score(X.iloc[train_idx], y.iloc[train_idx])
        val_score = pipeline.score(X.iloc[val_idx], y.iloc[val_idx])

        train_scores.append(train_score)
        val_scores.append(val_score)

        y_pred = pipeline.predict(X.iloc[val_idx])
        prec = precision_score(y.iloc[val_idx], y_pred)
        nPPV = (prec / remission_rate) * 100
        nPPVs.append(nPPV)

    print(f"Train Score: {np.mean(train_scores):.2f} ± {np.std(train_scores):.3f}")
    print(f"Validation Score: {np.mean(val_scores):.2f} ± {np.std(val_scores):.3f}")
    print(f"nPPV: {np.mean(nPPVs):.2f} ± {np.std(nPPVs):.3f}")

    return train_scores, val_scores, nPPVs

# try to replicate svc model results
svc_model = SVC(C=0.5)
train_scores, val_scores, nPPVs = MLpipe_shuffle_cv(X, y, preprocessor, svc_model, num_folds=1000)

Train Score: 0.67 ± 0.022
Validation Score: 0.64 ± 0.059
nPPV: 102.53 ± 4.166
